# Analytics — Plagiarism Detection Evaluation

This notebook evaluates the end-to-end pipeline for a single suspicious document against the **PAN 2011** ground-truth XML annotations.

## Pipeline Overview

| Stage | Description |
|-------|-------------|
| **1. Source Retrieval** | Four branches (TF-IDF, ESA, LSA, embeddings) fused into a ranked list. Candidates filtered by relative gap: keep docs scoring ≥ `top1_score × relative_gap`. |
| **2. LLM Source Confirmation** | LLM scores each candidate source doc (0–1). Docs below threshold (0.95) are dropped. |
| **3. Span Merging** | All chunk pairs from confirmed source docs are deduplicated to best-scoring match per suspicious chunk, then adjacent spans merged (gap ≤ MAX_GAP). |
| **4. Ground-Truth Evaluation** | Merged spans compared against XML GT using both binary overlap and PAN character-level metrics. |

## Evaluation Metrics

| Metric | Description |
|--------|-------------|
| **Binary span** | TP if any character overlap with a GT span (same source doc). Simple, intuitive. |
| **PAN char-level** | `char_precision = overlap_chars / det_chars`, `char_recall = overlap_chars / gt_chars`. Penalises over-merged spans. |

## Retrieval Analytics (Stage 1)

Shows the rank and relative gap of the true source document within the top-N candidates, so you can see whether the retrieval stage correctly surfaced the true source and how confidently it was separated from noise.

## Key Parameters

| Parameter | Default | Effect |
|-----------|---------|--------|
| `RELATIVE_GAP` | 0.70 | Keep candidates scoring ≥ `top1_score × 0.70` |
| `LLM_SCORE_THRESHOLD` | 0.95 | Min LLM confidence to keep a source doc |
| `TOP_PAIRS_PER_DOC` | 15 | Max pairs sent to LLM per source doc |
| `MAX_GAP` | 1800 chars | Max gap between chunks to merge into one span |

In [ ]:
import json, re, time
import pandas as pd
from pathlib import Path
from ollama import chat
from tqdm import tqdm

PROCESSED_DIR    = Path("../../../datasets/processed/PAN2011_300")
SUSPICIOUS_DOC_ID = "part1__suspicious-document00007.txt"

LLM_SCORE_THRESHOLD = 0.95
TOP_PAIRS_PER_DOC   = 15
MAX_GAP             = 1800
OLLAMA_MODEL        = "gemma4:e4b"

candidates_df     = pd.read_parquet(PROCESSED_DIR / "embedding_candidates_suspicious.parquet")
top20_df          = pd.read_parquet(Path("../04_source_retrieval/top20_df.parquet"))
source_chunks     = pd.read_parquet(PROCESSED_DIR / "source_chunks.parquet")
suspicious_chunks = pd.read_parquet(PROCESSED_DIR / "suspicious_chunks_embeddings.parquet")

top_source_ids = set(top20_df["source_doc_id"].tolist())
print(f"Retrieval candidates : {len(top_source_ids)} source docs")
print(top20_df[["source_doc_id", "final_score"]].to_string(index=False))

## Stage 1 — Retrieval Analytics

How well did the retrieval stage find the true source document?

- **Score distribution**: where does the true source rank vs. noise candidates?
- **Relative gap**: ratio of each candidate's score to the top-1 score (used for filtering)
- **Retrieval recall**: was the true source in the top-N?

In [ ]:
# ── Retrieval stage analytics ─────────────────────────────────────────────────
import xml.etree.ElementTree as ET

GT_XML_DIR = Path("../../../datasets/raw/PAN2011/pan-plagiarism-corpus-2011/external-detection-corpus/suspicious-documents")

def get_true_source_ids(suspicious_doc_id: str) -> list[str]:
    """Return source_doc_ids listed in the GT XML for this suspicious doc."""
    xml_name = suspicious_doc_id.replace(".txt", ".xml")
    # XML lives under part-specific subdir
    part = suspicious_doc_id.split("__")[0]
    xml_path = GT_XML_DIR / part / xml_name
    if not xml_path.exists():
        return []
    tree = ET.parse(xml_path)
    sources = []
    for feat in tree.findall(".//feature[@name='plagiarism']"):
        src = feat.get("source_reference", "")
        if src:
            sources.append(Path(src).name)
    return list(set(sources))


RELATIVE_GAP = 0.70  # same default as pipeline

true_sources = get_true_source_ids(SUSPICIOUS_DOC_ID)
print(f"Ground-truth source doc(s): {true_sources}")

# Add relative-gap column
top1_score = top20_df["final_score"].iloc[0]
top20_df = top20_df.copy()
top20_df["relative_gap"] = (top20_df["final_score"] / top1_score).round(4)
top20_df["is_true_source"] = top20_df["source_doc_id"].apply(
    lambda d: any(ts in d for ts in true_sources)
)

print(f"\nTop-1 score : {top1_score:.4f}")
print(f"Gap threshold (top1 * {RELATIVE_GAP}) : {top1_score * RELATIVE_GAP:.4f}")
print(f"\nRetrieval ranking:")
display_cols = ["source_doc_id", "final_score", "relative_gap", "is_true_source"]
if "weighted_mean" in top20_df.columns:
    display_cols.insert(2, "weighted_mean")
if "weighted_max" in top20_df.columns:
    display_cols.insert(3, "weighted_max")
print(top20_df[[c for c in display_cols if c in top20_df.columns]].to_string(index=True))

# Retrieval recall@K
kept_after_gap = top20_df[top20_df["relative_gap"] >= RELATIVE_GAP]
true_in_top20  = top20_df["is_true_source"].any()
true_in_gap    = kept_after_gap["is_true_source"].any()
true_rank      = top20_df.index[top20_df["is_true_source"]].tolist()

print(f"\n--- Retrieval Recall ---")
print(f"True source rank in top-{len(top20_df)}: {true_rank if true_rank else 'NOT FOUND'}")
print(f"Recall@{len(top20_df)}               : {'YES' if true_in_top20 else 'NO'}")
print(f"Candidates kept after gap filter: {len(kept_after_gap)} / {len(top20_df)}")
print(f"True source survives gap filter : {'YES' if true_in_gap else 'NO ← FALSE NEGATIVE at retrieval'}")

# Score gap visualisation (text bar chart)
print(f"\n--- Score distribution (■ = true source) ---")
for _, row in top20_df.iterrows():
    bar = "■" if row["is_true_source"] else "□"
    kept = "*" if row["relative_gap"] >= RELATIVE_GAP else " "
    print(f"  {kept}{bar} [{row.name:>2}] {row['source_doc_id'][:45]:<46}  score={row['final_score']:.4f}  gap={row['relative_gap']:.3f}")

In [ ]:
# ── Build chunk-pair candidates ───────────────────────────────────────────────
candidates_filtered = candidates_df[
    (candidates_df["suspicious_doc_id"] == SUSPICIOUS_DOC_ID) &
    (candidates_df["source_doc_id"].isin(top_source_ids))
].copy()

susp_text = (
    suspicious_chunks[suspicious_chunks["doc_id"] == SUSPICIOUS_DOC_ID]
    [["chunk_id", "embedding_text"]]
    .rename(columns={"chunk_id": "suspicious_chunk_id", "embedding_text": "suspicious_text"})
)

src_text = (
    source_chunks[source_chunks["doc_id"].isin(top_source_ids)]
    [["chunk_id", "chunk_text"]]
    .rename(columns={"chunk_id": "source_chunk_id", "chunk_text": "source_text"})
)

pairs_df = (
    candidates_filtered
    .merge(susp_text, on="suspicious_chunk_id", how="inner")
    .merge(src_text,  on="source_chunk_id",     how="inner")
)

print(f"Total chunk pairs : {len(pairs_df)}")
print(f"Source docs       : {pairs_df['source_doc_id'].nunique()}")

In [ ]:
# ── LLM source confirmation ───────────────────────────────────────────────────
def _parse_json(raw: str) -> dict:
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON in response: {raw[:300]}")
    clean = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', match.group())
    return json.loads(clean)


def score_source_doc(source_doc_id: str, pairs: list[dict]) -> dict:
    pairs_text = "\n\n".join([
        f"[Pair {i+1}]\n"
        f"SUSPICIOUS: {p['suspicious_text'][:600]}\n"
        f"SOURCE CANDIDATE: {p['source_text'][:600]}"
        for i, p in enumerate(pairs)
    ])
    prompt = (
        f"You are a plagiarism detection expert.\n"
        f"Below are {len(pairs)} text pair(s). Each pair shows a chunk from a SUSPICIOUS document "
        f"alongside a chunk from a CANDIDATE SOURCE document.\n\n"
        f"{pairs_text}\n\n"
        f"Analyze whether the suspicious chunks appear to be copied, paraphrased, or otherwise "
        f"derived from the source document. "
        f"Score the overall likelihood that this source document is the true origin of the "
        f"suspicious text (0.0 = definitely not, 1.0 = definitely yes). "
        f"Respond with ONLY a JSON object — no markdown, no explanation — with keys: "
        f"score (float 0-1), is_likely_source (bool), reasoning (string)."
    )
    t0 = time.time()
    response = chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
        think=False,
    )
    data = _parse_json(response.message.content)
    return {
        "source_doc_id":        source_doc_id,
        "llm_score":            float(data.get("score", 0.0)),
        "llm_is_likely_source": bool(data.get("is_likely_source", False)),
        "llm_reasoning":        data.get("reasoning", ""),
        "elapsed_s":            round(time.time() - t0, 1),
    }


top_pairs = (
    pairs_df
    .sort_values("embedding_score", ascending=False)
    .groupby("source_doc_id")
    .head(TOP_PAIRS_PER_DOC)
    .reset_index(drop=True)
)

llm_rows = []
for source_doc_id, group in tqdm(top_pairs.groupby("source_doc_id"), desc="LLM scoring"):
    pairs = group[["suspicious_text", "source_text", "embedding_score"]].to_dict("records")
    try:
        llm_rows.append(score_source_doc(source_doc_id, pairs))
    except Exception as e:
        print(f"[WARN] {source_doc_id}: {e}")
        llm_rows.append({"source_doc_id": source_doc_id, "llm_score": 0.0,
                         "llm_is_likely_source": False, "llm_reasoning": f"Error: {e}", "elapsed_s": 0.0})

llm_scores_df = pd.DataFrame(llm_rows).sort_values("llm_score", ascending=False).reset_index(drop=True)
confirmed_ids = set(llm_scores_df.loc[llm_scores_df["llm_score"] >= LLM_SCORE_THRESHOLD, "source_doc_id"])

print(f"Confirmed source docs (score >= {LLM_SCORE_THRESHOLD}): {len(confirmed_ids)}")
llm_scores_df[["source_doc_id", "llm_score", "llm_is_likely_source", "elapsed_s"]]

In [ ]:
# ── Span merging ─────────────────────────────────────────────────────────────
confirmed_pairs = pairs_df[pairs_df["source_doc_id"].isin(confirmed_ids)].copy()

# Deduplicate: keep highest embedding_score match per suspicious chunk
best = (
    confirmed_pairs
    .sort_values("embedding_score", ascending=False)
    .drop_duplicates(subset=["suspicious_chunk_id"], keep="first")
    .sort_values(["source_doc_id", "suspicious_start_char"])
    .reset_index(drop=True)
)

merged_rows = []
for _, grp in best.groupby("source_doc_id"):
    grp = grp.reset_index(drop=True)
    current = grp.iloc[0].to_dict()
    for _, row in grp.iloc[1:].iterrows():
        gap = row["suspicious_start_char"] - current["suspicious_end_char"]
        if gap <= MAX_GAP:
            current["suspicious_end_char"] = max(current["suspicious_end_char"], row["suspicious_end_char"])
            current["source_start_char"]   = min(current["source_start_char"],   row["source_start_char"])
            current["source_end_char"]     = max(current["source_end_char"],     row["source_end_char"])
            if row["embedding_score"] > current["embedding_score"]:
                current["embedding_score"] = row["embedding_score"]
        else:
            merged_rows.append(current)
            current = row.to_dict()
    merged_rows.append(current)

detected = pd.DataFrame(merged_rows).reset_index(drop=True)

print(f"Confirmed pairs (all chunks) : {len(confirmed_pairs)}")
print(f"After dedup (1 per chunk)    : {len(best)}")
print(f"After span merging           : {len(detected)}")
print()
detected[["source_doc_id", "suspicious_start_char", "suspicious_end_char", "embedding_score"]]

In [ ]:
# ── Ground-truth evaluation ───────────────────────────────────────────────────
GROUND_TRUTH_PATH = Path("../../../datasets/processed/PAN2011_ground_truth/pan2011_plagiarism_spans.parquet")
gt_df  = pd.read_parquet(GROUND_TRUTH_PATH)
gt_doc = gt_df[gt_df["suspicious_doc_id"] == SUSPICIOUS_DOC_ID].copy()

def char_overlap(a_start, a_end, b_start, b_end) -> int:
    return max(0, min(a_end, b_end) - max(a_start, b_start))

# ── Binary span metric ────────────────────────────────────────────────────────
hit_flags, matched_gt = [], set()
total_overlap_chars = total_det_chars = 0

for _, det in detected.iterrows():
    hit = False
    det_len = det["suspicious_end_char"] - det["suspicious_start_char"]
    total_det_chars += det_len
    for gt_idx, gt in gt_doc.iterrows():
        if det["source_doc_id"] != gt["source_doc_id"]:
            continue
        ov = char_overlap(det["suspicious_start_char"], det["suspicious_end_char"],
                          gt["suspicious_offset"],      gt["suspicious_end"])
        if ov > 0:
            hit = True
            matched_gt.add(gt_idx)
            total_overlap_chars += ov
    hit_flags.append(hit)

detected = detected.copy()
detected["matched_gt"] = hit_flags

total_gt_chars = int((gt_doc["suspicious_end"] - gt_doc["suspicious_offset"]).sum())

tp = sum(hit_flags)
fp = len(detected) - tp
fn = len(gt_doc) - len(matched_gt)

precision = tp / (tp + fp)           if (tp + fp) else 0.0
recall    = tp / (tp + fn)           if (tp + fn) else 0.0
f1        = 2*precision*recall / (precision+recall) if (precision+recall) else 0.0

char_p  = total_overlap_chars / total_det_chars  if total_det_chars  else 0.0
char_r  = total_overlap_chars / total_gt_chars   if total_gt_chars   else 0.0
char_f1 = 2*char_p*char_r / (char_p+char_r)     if (char_p+char_r)  else 0.0

detected["det_susp_len"] = detected["suspicious_end_char"] - detected["suspicious_start_char"]
gt_doc["gt_susp_len"]    = gt_doc["suspicious_end"] - gt_doc["suspicious_offset"]

print(f"GT spans : {len(gt_doc)}   Detected spans : {len(detected)}")
print(f"TP={tp}  FP={fp}  FN={fn}")
print()
print(f"{'Metric':<18} {'Binary':>8} {'Char-level':>12}")
print(f"{'Precision':<18} {precision:>8.4f} {char_p:>12.4f}")
print(f"{'Recall':<18} {recall:>8.4f} {char_r:>12.4f}")
print(f"{'F1':<18} {f1:>8.4f} {char_f1:>12.4f}")
print()
print(f"Detected chars  : {total_det_chars:,}   GT chars : {total_gt_chars:,}   Overlap : {total_overlap_chars:,}  ({100*char_r:.1f}% of GT covered)")
print()
print("=== Span size: detected vs GT (suspicious side) ===")
print(f"Detected — min={detected['det_susp_len'].min():.0f}  median={detected['det_susp_len'].median():.0f}  max={detected['det_susp_len'].max():.0f}")
print(f"GT       — min={gt_doc['gt_susp_len'].min():.0f}  median={gt_doc['gt_susp_len'].median():.0f}  max={gt_doc['gt_susp_len'].max():.0f}")